# Silver Layer, Products Data Processing
**Source:** Bronze Delta Table `bronze.bronze_products`  
**Target:** Silver Delta Table `/delta/silver/products` (Hive table `silver.silver_products`)  
**Pattern:** Incremental Load with Merge (UPSERT) based on `product_id`  
**Transformations:**
- Normalize price, stock_quantity, rating (clamp negative/out-of-range values)
- Derive `price_category` (Premium/Standard/Budget) from `price`
- Derive `stock_status` from `stock_quantity`
- Filter out records with null `name` or `category`
- Add `last_updated` timestamp
  
**Layer:** Silver

In [3]:
import os, sys, logging
from pyspark.sql import SparkSession

os.environ["SPARK_HOME"] = "/opt/spark"
os.environ["PYTHONPATH"] = "/opt/spark/python:/opt/spark/python/lib/py4j-0.10.9.7-src.zip"
sys.path.insert(0, "/opt/spark/python")
sys.path.insert(0, "/opt/spark/python/lib/py4j-0.10.9.7-src.zip")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

CONFIG = {
    "bronze_db"      : "bronze",
    "bronze_table"   : "bronze_products",
    "silver_db"      : "silver",
    "silver_table"   : "silver_products",
    "silver_path"    : "hdfs://master:8020/delta/silver/products",
}

def get_spark():
    return (
        SparkSession.builder
        .appName("Silver_Products_Load")
        .master("spark://master:7077")
        .config("spark.jars.packages",
                "io.delta:delta-spark_2.12:3.2.1")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.hadoop.fs.defaultFS", "hdfs://master:8020")
        .config("spark.sql.warehouse.dir", "hdfs://master:8020/user/hive/warehouse")
        .config("spark.hadoop.hive.metastore.uris", "thrift://master:9083")
        .config("spark.databricks.delta.stats.collect", "false")
        .enableHiveSupport()
        .getOrCreate()
    )

def create_silver_table_if_not_exists(spark):
    """Create the silver products table if it doesn't exist."""
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {CONFIG['silver_db']}")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {CONFIG['silver_db']}.{CONFIG['silver_table']} (
            product_id STRING,
            name STRING,
            category STRING,
            brand STRING,
            price DOUBLE,
            stock_quantity INT,
            rating DOUBLE,
            is_active BOOLEAN,
            price_category STRING,
            stock_status STRING,
            last_updated TIMESTAMP
        )
        USING DELTA
        LOCATION '{CONFIG['silver_path']}'
    """)
    logger.info(f"Ensured table {CONFIG['silver_db']}.{CONFIG['silver_table']} exists.")

def get_last_processed_timestamp(spark):
    """Return the maximum last_updated timestamp from silver table, or a default."""
    try:
        df = spark.sql(f"SELECT MAX(last_updated) AS last_processed FROM {CONFIG['silver_db']}.{CONFIG['silver_table']}")
        row = df.collect()[0]
        ts = row['last_processed']
        if ts is None:
            return "1900-01-01T00:00:00.000+00:00"
        else:
            return ts.isoformat() if hasattr(ts, 'isoformat') else str(ts)
    except Exception as e:
        logger.warning(f"Could not read last_processed (table may be empty): {e}")
        return "1900-01-01T00:00:00.000+00:00"

def run():
    logger.info("Silver Products pipeline started.")
    spark = get_spark()
    spark.sparkContext.setLogLevel("ERROR")

    create_silver_table_if_not_exists(spark)

    last_processed = get_last_processed_timestamp(spark)
    logger.info(f"Last processed timestamp: {last_processed}")

    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW bronze_incremental AS
        SELECT *
        FROM {CONFIG['bronze_db']}.{CONFIG['bronze_table']}
        WHERE ingestion_timestamp > TIMESTAMP '{last_processed}'
    """)

    count_new = spark.table("bronze_incremental").count()
    if count_new == 0:
        logger.info("No new data to process. Exiting.")
        return
    logger.info(f"Found {count_new} new rows in bronze.")

    spark.sql("""
        CREATE OR REPLACE TEMP VIEW silver_incremental AS
        SELECT
            product_id,
            name,
            category,
            brand,
            CASE WHEN price < 0 THEN 0 ELSE price END AS price,
            CASE WHEN stock_quantity < 0 THEN 0 ELSE stock_quantity END AS stock_quantity,
            CASE
                WHEN rating < 0 THEN 0
                WHEN rating > 5 THEN 5
                ELSE rating
            END AS rating,
            is_active,
            CASE
                WHEN price > 1000 THEN 'Premium'
                WHEN price > 100 THEN 'Standard'
                ELSE 'Budget'
            END AS price_category,
            CASE
                WHEN stock_quantity = 0 THEN 'Out of Stock'
                WHEN stock_quantity < 10 THEN 'Low Stock'
                WHEN stock_quantity < 50 THEN 'Moderate Stock'
                ELSE 'Sufficient Stock'
            END AS stock_status,
            CURRENT_TIMESTAMP() AS last_updated
        FROM bronze_incremental
        WHERE name IS NOT NULL AND category IS NOT NULL
    """)

    logger.info("Sample of transformed silver data:")
    spark.table("silver_incremental").show(5, truncate=False)

    # Merge into silver table
    merge_result = spark.sql(f"""
        MERGE INTO {CONFIG['silver_db']}.{CONFIG['silver_table']} AS target
        USING silver_incremental AS source
        ON target.product_id = source.product_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    logger.info(f"Merge completed: {merge_result.first()['num_affected_rows']} rows affected.")
    final_count = spark.table(f"{CONFIG['silver_db']}.{CONFIG['silver_table']}").count()
    logger.info(f"SUCCESS: Silver products table now has {final_count} rows.")

if __name__ == "__main__":
    run()

2026-03-19 12:58:55,470 [INFO] Silver Products pipeline started.


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jupyter/.ivy2/cache
The jars for the packages stored in: /home/jupyter/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e82790d4-290a-4b91-a0b5-f45c69561c3a;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.1/delta-spark_2.12-3.2.1.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.1!delta-spark_2.12.jar (4418ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.1/delta-storage-3.2.1.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.1!delta-storage.jar (141ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (369ms)
:: resolution report :: resolve 3683ms

+----------+---------+-----------+------------+------+--------------+------+---------+--------------+----------------+-----------------------+
|product_id|name     |category   |brand       |price |stock_quantity|rating|is_active|price_category|stock_status    |last_updated           |
+----------+---------+-----------+------------+------+--------------+------+---------+--------------+----------------+-----------------------+
|1         |Product 1|Toys       |BeautyGlow  |995.73|989           |3.5   |true     |Standard      |Sufficient Stock|2026-03-19 12:59:57.383|
|2         |Product 2|Garden     |GardenMaster|497.76|495           |3.8   |true     |Standard      |Sufficient Stock|2026-03-19 12:59:57.383|
|3         |Product 3|Electronics|BeautyGlow  |331.63|10            |4.6   |true     |Standard      |Moderate Stock  |2026-03-19 12:59:57.383|
|4         |Product 4|Beauty     |TechPro     |798.83|683           |4.7   |false    |Standard      |Sufficient Stock|2026-03-19 12:59:57.383|

2026-03-19 13:00:06,754 [INFO] Merge completed: 1000 rows affected.             
2026-03-19 13:00:11,068 [INFO] SUCCESS: Silver products table now has 1000 rows.
